# Import

In [1]:
%pip install tensorflow keras transformer


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement transformer (from versions: none)
ERROR: No matching distribution found for transformer


In [2]:
%pip install keras-tuner


Note: you may need to restart the kernel to use updated packages.


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
%pip install hmmlearn
%pip install pgmpy

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install keras-nlp --upgrade

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install tf-keras

In [7]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tqdm import tqdm
import keras_tuner as kt
from tensorflow.keras.models import load_model
import keras_nlp

e:\anaconda3\envs\ml_env_test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score

In [9]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import MaxAbsScaler, MinMaxScaler
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import hmmlearn.hmm
from hmmlearn.hmm import GaussianHMM
from sklearn_crfsuite import CRF
from sklearn.metrics import log_loss, hinge_loss, precision_score, recall_score, f1_score, roc_auc_score

In [10]:
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

In [11]:
## Options
pd.set_option("max_colwidth", None)

In [12]:
# Get the absolute path to the 'src' directory
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(project_root)
print(project_root)

e:\2_LEARNING_BKU\2_File_2\K22_HK242\CO3117_Machine_Learning\Main


In [13]:
from src.features.build_features_utils import *  # Assuming build_features_utils is inside build_features.py
from src.models.models_utils import *  # Assuming utils.py exists inside src/models/

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Dict

In [14]:
# Dictionary for models
MODEL_DICT = {
    "decision_tree": DecisionTreeClassifier,
    "perceptron": Perceptron,
    "mlp": MLPClassifier,
    "bayesian": GaussianNB,
    "random_forest": RandomForestClassifier,
    "xgboost": xgb.XGBClassifier,
    "logistic_regression": LogisticRegression,
    "svm": SVC,
    "lda": LDA
} 

# Dictionary for model parameters
MODEL_PARAMS = {
    "lda": {
        "solver": ["lsqr", "eigen"],
        "shrinkage": [None, "auto"],  # Only used with 'lsqr' or 'eigen'
        "tol": [1e-4, 1e-3, 1e-2]     # Tolerance for convergence
    },
    
    # "decision_tree": {
    #     "criterion": ["gini", "entropy"],
    #     "max_depth": [10, 20],
    #     "min_samples_split": [2, 5],
    #     "min_samples_leaf": [1, 2],
    #     "max_features": ["sqrt", "log2"]
    # },
    
    "decision_tree": {
        "criterion": ["gini", "entropy"],
        "max_depth": [10, 20, 30, 40],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    },
    
    # "perceptron": {
    #     "max_iter": [1000, 2000],
    #     "tol": [1e-3],
    #     "eta0": [0.001],
    #     "penalty": ["l2"],
    #     "alpha": [0.0001, 0.001]
    # },
    
    "perceptron": {
        "max_iter": [1000, 2000],
        "tol": [1e-3, 1e-4],
        "eta0": [0.001, 0.01, 0.1],
        "penalty": [None, "l2", "l1"],
        "alpha": [0.0001, 0.001, 0.01]
    },
    
    "mlp": {
        "hidden_layer_sizes": [(100,)],
        "activation": ["tanh", "logistic"],
        "solver": ["sgd"],
        "alpha": [0.01],
        "batch_size": [32],
        "max_iter": [2000],
    },
    
    # "mlp": {
    #     "hidden_layer_sizes": [(50,), (100,), (50, 50), (100, 100)],
    #     "activation": ["relu", "tanh", "logistic"],
    #     "solver": ["adam", "sgd"],
    #     "alpha": [0.0001, 0.001, 0.01],
    #     "batch_size": [32, 64, 128],
    #     "max_iter": [500, 1000],
    #     "learning_rate": ["constant", "invscaling", "adaptive"]
    # },
    
    "bayesian": {
        "priors": [None, [0.5, 0.5], [0.4, 0.6], [0.3, 0.7], [0.2, 0.8], [0.1, 0.9], [0.05, 0.95]],
        "var_smoothing": [1e-9, 1e-8, 1e-7]
    },
    
    "random_forest": {
        "n_estimators": [100, 200],
        "max_depth": [10],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"],
        "bootstrap": [True, False]
    },
    
    # "random_forest": {
    #     "n_estimators": [50, 100, 200],
    #     "max_depth": [None, 10, 20, 30],
    #     "min_samples_split": [2, 5, 10],
    #     "min_samples_leaf": [1, 2, 4],
    #     "max_features": ["auto", "sqrt", "log2"],
    #     "bootstrap": [True, False]
    # },
    
    "xgboost": {
        "n_estimators": [100],
        "learning_rate": [0.01, 0.1],
        "max_depth": [6, 10]
    },
    
    # "xgboost": {
    #     "n_estimators": [100, 200, 300],
    #     "learning_rate": [0.01, 0.1, 0.2],
    #     "max_depth": [3, 6, 10],
    #     "subsample": [0.8, 1.0],
    #     "colsample_bytree": [0.8, 1.0],
    #     "gamma": [0, 0.1, 0.2]
    # },
    
    "svm": {
        "kernel": ["linear"],
        "C": [0.001, 0.01, 0.1, 1],
        "gamma": [0.1, 0.01, "scale", "auto"]
    },
    
    # "svm": {
    #     "kernel": ["linear", "rbf", "poly"],
    #     "C": [0.1, 1, 10, 100],
    #     "gamma": [0.1, 0.01, "scale", "auto"],
    #     "degree": [2, 3, 4]
    # },
    
    # "logistic_regression": {
    #     "penalty": ["l2"],
    #     "C": [0.1, 1.0],
    #     "max_iter": [1000, 2000]
    # },
    
    "logistic_regression": {
        "penalty": ["l1", "l2", "elasticnet", None],
        "C": [0.1, 1.0, 10.0],
        "max_iter": [1000, 2000]
    },
    
    
    # "hmm": {
    #     "n_components": [2],  # Keep it small
    #     "covariance_type": ["diag"],  # Simpler covariance type
    #     "n_iter": [500],  # Reduce iterations
    #     "init_params": ["stmc"],  # Initialize start probabilities, transition matrix, and means/covariance
    #     "params": ["stmc"]
    # },
    
    "hmm": {
        "n_components": [2, 3, 4],
        "covariance_type": ["diag", "full", "tied"],
        "n_iter": [100, 200],
        "init_params": ["c", "s", "cs"],
        "params": ["c", "t", "ct"]
    },
    
    "bayes_network": {
        "structure": [None],
        "n_bins": [2],
        "strategy": ["kmeans"],
        "min_unique_values": [2],
        "max_features": [10]
    },
    
    # "crf": {
    #     "c1": [0.1, 0.01],  # L1 Regularization
    #     "c2": [0.1, 0.01],  # L2 Regularization
    #     "max_iterations": [50, 100]  # Limit iterations
    # }
}

BEST_MODEL_PARAMS = {
    "decision_tree": {
        "criterion": "gini",
        "max_depth": 40,
        "min_samples_split": 10,
        "min_samples_leaf": 4,
        "max_features": "sqrt"
    },
    
    "perceptron": {
        "max_iter": 1000,
        "tol": 1e-3,
        "eta0": 0.001,
        "penalty": "l2",
        "alpha": 0.0001
    },
    
    "mlp": {
        "hidden_layer_sizes": (100,),
        "activation": "logistic",
        "solver": "sgd",
        "alpha": 0.01,
        "batch_size": 32,
        "max_iter": 2000,
    },
    
    "bayesian": {
        "priors": [0.3, 0.7],
        "var_smoothing": 1e-9
    },
    
    "random_forest": {
        "n_estimators": [100],
        "max_depth": [10],
        "min_samples_split": [5],
        "min_samples_leaf": [1],
        "max_features": ["sqrt"]
    },
    
    "xgboost": {
        "n_estimators": 150,
        "learning_rate": 0.1,
        "max_depth": 15
    },
    
    "svm": {
        "kernel": ["linear"],
        "C": [0.001, 0.01, 0.1, 1],
        "gamma": [0.1, 0.01, "scale", "auto"]
    },
    
    "logistic_regression": {
        "penalty": "l2",
        "C": 0.1,
        "max_iter": 1000
    },
    
    "hmm": {
        "n_components": [2, 3, 4],
        "covariance_type": ["diag", "full", "tied"],
        "n_iter": [100, 200],
        "init_params": ["c", "s", "cs"],
        "params": ["c", "t", "ct"]
    },
    
    "bayes_network": {
        "structure": [None],
        "n_bins": [2],
        "strategy": ["kmeans"],
        "min_unique_values": [2],
        "max_features": [10]
    },
    
    # "crf": {
    #     "c1": [0.1, 0.01],  # L1 Regularization
    #     "c2": [0.1, 0.01],  # L2 Regularization
    #     "max_iterations": [50, 100]  # Limit iterations
    # }
}

# Dictionary for dimensionality reduction methods
DIMENSIONALITY_REDUCTION_DICT = {
    "pca": PCA,
    "lda": LDA,
}

# Load dataset

In [15]:
# Load dataset
dataset_path = os.path.join(project_root, "data", "final", "final_clean_no_neutral_no_duplicates_v1.csv")
df = pd.read_csv(dataset_path)


In [16]:
df.head()

,target,text,text_clean,text_length,text_clean_length
0,0.0,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D",switchfoot awww thats bummer shoulda got david carr third day,19,10
1,0.0,is upset that he can't update his Facebook by texting it... and might cry as a result School today also. Blah!,upset cant update facebook texting might cry result school today also blah,21,12
2,0.0,@Kenichan I dived many times for the ball. Managed to save 50% The rest go out of bounds,kenichan dived many times ball managed save rest go bounds,18,10
3,0.0,my whole body feels itchy and like its on fire,whole body feels itchy like fire,10,6
4,0.0,"@nationwideclass no, it's not behaving at all. i'm mad. why am i here? because I can't see you all over there.",nationwideclass behaving im mad cant see,21,6


In [17]:
# Replace target 4 with 1
df["target"] = df["target"].replace(4, 1)


# Build features

## Defined

In [18]:
# feature_methods = ["tfidf", "count", "word2vec", "glove"]
feature_methods = ["count"]
df_sampled = df.sample(n=1000, random_state=42)

In [19]:
doc_lst = df_sampled["text_clean"].tolist()
label_lst = df_sampled["target"].tolist()

In [20]:
X_train_features_dict, X_test_features_dict, y_train, y_test = build_vector_for_text(df_sampled, feature_methods, project_root)


🔎 Running feature extraction...



Feature Extraction Progress: 100%|██████████| 1/1 [00:00<00:00,  8.26it/s]


🔍 Processing feature extraction using: count...
✅ count - Train shape: (800, 2000), Test shape: (200, 2000)


In [21]:
print("\n📊 Dataset Shapes:")

# Print the shape of feature matrices for each feature method
for feature_method, X_train in X_train_features_dict.items():
    print(f"🔹 X_train ({feature_method}): {X_train.shape}")
    
for feature_method, X_test in X_test_features_dict.items():
    print(f"🔹 X_test ({feature_method}): {X_test.shape}")

# Print y_train and y_test shapes
print(f"\n🎯 y_train shape: {y_train.shape}")
print(f"🎯 y_test shape: {y_test.shape}")



📊 Dataset Shapes:
🔹 X_train (count): (800, 2000)
🔹 X_test (count): (200, 2000)

🎯 y_train shape: (800,)
🎯 y_test shape: (200,)


## Test PCA (reduce dim) - LDA (classifier) => done

In [22]:
# X_train_pca_dict, X_test_pca_dict, y_train, y_test = build_vector_for_text(df_sampled, feature_methods, project_root, "pca", 100)

In [23]:
# print("\n📊 Dataset Shapes:")

# # Print the shape of feature matrices for each feature method
# for feature_method, X_train in X_train_pca_dict.items():
#     print(f"🔹 X_train ({feature_method}): {X_train.shape}")
    
# for feature_method, X_test in X_test_pca_dict.items():
#     print(f"🔹 X_test ({feature_method}): {X_test.shape}")

# # Print y_train and y_test shapes
# print(f"\n🎯 y_train shape: {y_train.shape}")
# print(f"🎯 y_test shape: {y_test.shape}")


## Test cho cac api call moi trong feature builder => done

In [24]:
# # Test each feature selection method
# feature_selections = ["variance", "chi2", "topic_modeling", None]
# for feature_selection in feature_selections:
#     print(f"\n🔬 Testing with feature_selection: {feature_selection or 'None'}")
    
#     # Build features with current feature selection method
#     X_train_features_dict, X_test_features_dict, y_train, y_test = build_vector_for_text(
#         df_sampled=df_sampled,
#         feature_methods=feature_methods,
#         project_root=project_root,
#         reduce_dim="pca",  # Adding PCA as an example, can be None or "lda"
#         n_components=50,
#         feature_selection=feature_selection
#     )

#     print("\n📊 Dataset Shapes:")
    
#     # Print the shape of feature matrices for each feature method
#     for feature_method, X_train in X_train_features_dict.items():
#         print(f"🔹 X_train ({feature_method}): {X_train.shape}")
        
#     for feature_method, X_test in X_test_features_dict.items():
#         print(f"🔹 X_test ({feature_method}): {X_test.shape}")

#     # Print y_train and y_test shapes
#     print(f"\n🎯 y_train shape: {y_train.shape}")
#     print(f"🎯 y_test shape: {y_test.shape}")
    
#     print("-" * 50)

Variance 

=> count & glove 

=> not tfidf because No feature in X meets the variance threshold 0.01000. Skipping this method.

=> not w2v as n_components=50 must be between 0 and min(n_samples, n_features)=1 with svd_solver='covariance_eigh'. Skipping this method.

chi2 

=> can be with tfidf and count 

=> not w2v + glove as Input X must be non-negative.. Skipping this method.

topic_modeling

=> can be with tfidf and count 

=> not w2v + glove as Negative values in data passed to LatentDirichletAllocation.fit

Negative values in data passed to LatentDirichletAllocation.fit

## Test the Voting Classifier 

In [25]:
SELECTED_MODEL_DICT = {
    "logistic_regression": LogisticRegression,
    "xgboost": xgb.XGBClassifier,
    "mlp": MLPClassifier,
    "bayesian": GaussianNB,
}

In [ ]:
# Call the function with all models (one of each)
feature_use = "count"

voting_clf = train_voting_classifier(
    model_dict=SELECTED_MODEL_DICT, 
    param_dict=BEST_MODEL_PARAMS, 
    feature_method=feature_use, 
    X=X_train_features_dict[feature_use], 
    y=y_train, 
    voting_type='soft', 
    model_save_path="voting_model_all_models.pkl"
)


In [28]:
# Load the trained VotingClassifier model from the file
voting_clf = joblib.load("voting_model_all_models.pkl")
print("✅ Model loaded successfully from 'voting_model_all_models.pkl'")

✅ Model loaded successfully from 'voting_model_all_models.pkl'


In [29]:
# Test the trained model on the test set
y_pred = voting_clf.predict(X_test_features_dict[feature_use])

# Print evaluation metrics
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")

# Print Classification Report
print("\n📄 Classification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.6550
ROC AUC: 0.6565
F1 Score: 0.6634
Precision: 0.6939
Recall: 0.6355

📄 Classification Report:

              precision    recall  f1-score   support

         0.0       0.62      0.68      0.65        93
         1.0       0.69      0.64      0.66       107

    accuracy                           0.66       200
   macro avg       0.66      0.66      0.65       200
weighted avg       0.66      0.66      0.66       200



In [31]:
# Correct way to create multiple Logistic Regression models (passing classes, not objects)
multiple_lr_models = {
    f"logistic_regression_{i}": LogisticRegression  # Only pass the class, not the instance
    for i in range(5)
}

# Provide a parameter dictionary
lr_params = {
    f"logistic_regression_{i}": {"penalty": "l2", "C": 0.1, "max_iter": 1000, "random_state": i}
    for i in range(5)
}

# Call the function with multiple logistic regression models
voting_clf = train_voting_classifier(
    model_dict=multiple_lr_models, 
    param_dict=lr_params,  # Now we provide the parameter dictionary
    feature_method=feature_use, 
    X=X_train_features_dict[feature_use], 
    y=y_train, 
    voting_type='soft', 
    model_save_path="voting_model_multiple_lr.pkl"
)

# Test the trained model on the test set
y_pred = voting_clf.predict(X_test_features_dict[feature_use])

# Print evaluation metrics
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")

# Print Classification Report
print("\n📄 Classification Report:\n")
print(classification_report(y_test, y_pred))


🚀 Training Voting Classifier (soft) with feature method: count


🎯 Running K-Fold Cross-Validation...


K-Fold Progress: 100%|██████████| 5/5 [00:03<00:00,  1.28it/s]


📊 Avg Accuracy: 0.6575
📊 Avg ROC AUC: 0.6488
📊 Avg F1 Score: 0.7198
📊 Avg Precision: 0.6389
📊 Avg Recall: 0.8311
💾 Model saved to voting_model_multiple_lr.pkl
Accuracy: 0.6050
ROC AUC: 0.5858
F1 Score: 0.6996
Precision: 0.5897
Recall: 0.8598

📄 Classification Report:

              precision    recall  f1-score   support

         0.0       0.66      0.31      0.42        93
         1.0       0.59      0.86      0.70       107

    accuracy                           0.60       200
   macro avg       0.62      0.59      0.56       200
weighted avg       0.62      0.60      0.57       200



## Test Stacking

In [32]:
# Define multiple Logistic Regression models (as classes, not instances)
multiple_lr_models = {
    f"logistic_regression_{i}": LogisticRegression  # Note: Passing the class, not instance
    for i in range(5)
}

# Provide parameters for each logistic regression model
lr_params = {
    f"logistic_regression_{i}": {"penalty": "l2", "C": 0.1, "max_iter": 1000, "random_state": i}
    for i in range(5)
}

# Train Stacking Classifier
stacking_clf = train_stacking_classifier(
    model_dict=multiple_lr_models,
    param_dict=lr_params,
    feature_method=feature_use,
    X=X_train_features_dict[feature_use],
    y=y_train,
    final_estimator=LogisticRegression(),
    model_save_path="stacking_model_multiple_lr.pkl"
)

# Test the trained model on the test set
y_pred = stacking_clf.predict(X_test_features_dict[feature_use])

# Print evaluation metrics
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")

# Print Classification Report
print("\n📄 Classification Report:\n")
print(classification_report(y_test, y_pred))


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



🚀 Training Stacking Classifier with feature method: count


🎯 Running K-Fold Cross-Validation...


K-Fold Progress: 100%|██████████| 5/5 [00:26<00:00,  5.38s/it]


📊 Avg Accuracy: 0.6437
📊 Avg ROC AUC: 0.6375
📊 Avg F1 Score: 0.6959
📊 Avg Precision: 0.6387
📊 Avg Recall: 0.7693
💾 Model saved to stacking_model_multiple_lr.pkl
Accuracy: 0.6250
ROC AUC: 0.6094
F1 Score: 0.7036
Precision: 0.6096
Recall: 0.8318

📄 Classification Report:

              precision    recall  f1-score   support

         0.0       0.67      0.39      0.49        93
         1.0       0.61      0.83      0.70       107

    accuracy                           0.62       200
   macro avg       0.64      0.61      0.60       200
weighted avg       0.64      0.62      0.60       200



In [33]:
stacking_clf = train_stacking_classifier(
    model_dict=SELECTED_MODEL_DICT,
    param_dict=BEST_MODEL_PARAMS,
    feature_method=feature_use,
    X=X_train_features_dict[feature_use],
    y=y_train,
    final_estimator=LogisticRegression(),
    model_save_path="stacking_model_selected_models.pkl"
)

# Test the trained model on the test set
y_pred = stacking_clf.predict(X_test_features_dict[feature_use])

# Print evaluation metrics
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")

# Print Classification Report
print("\n📄 Classification Report:\n")
print(classification_report(y_test, y_pred))


🚀 Training Stacking Classifier with feature method: count


🎯 Running K-Fold Cross-Validation...


K-Fold Progress:   0%|          | 0/5 [00:00<?, ?it/s]e:\anaconda3\envs\ml_env_test\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\anaconda3\envs\ml_env_test\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\anaconda3\envs\ml_env_test\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
K-Fold Progress:  40%|████      | 2/5 [28:32<36:15, 725.01s/it]   e:\anaconda3\envs\ml_env_test\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20

📊 Avg Accuracy: 0.6387
📊 Avg ROC AUC: 0.6413
📊 Avg F1 Score: 0.6630
📊 Avg Precision: 0.6612
📊 Avg Recall: 0.6745


e:\anaconda3\envs\ml_env_test\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
e:\anaconda3\envs\ml_env_test\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(


💾 Model saved to stacking_model_selected_models.pkl
Accuracy: 0.6350
ROC AUC: 0.6251
F1 Score: 0.6920
Precision: 0.6308
Recall: 0.7664

📄 Classification Report:

              precision    recall  f1-score   support

         0.0       0.64      0.48      0.55        93
         1.0       0.63      0.77      0.69       107

    accuracy                           0.64       200
   macro avg       0.64      0.63      0.62       200
weighted avg       0.64      0.64      0.63       200



# New API Call

In [ ]:
model_name_lst = [
    # "decision_tree",
    # "random_forest", 
    # "xgboost", 
    # "perceptron", 
    # "mlp", 
    # "lstm",
    # "bayesian",
    # "GA",
    # "hmm",
    # "bayesnet",
    # "logistic_regression",
    # "svm",
    # "lda",
    "bilstm",
    "bert"
]

In [ ]:
# trained_model = os.path.join(project_root, "src", "models")

In [ ]:
# %pip uninstall tf-nightly
# %pip install tensorflow


In [ ]:
# %pip show keras-nlp


In [ ]:
# train_general_model(df_sampled, doc_lst, label_lst, model_name_lst, feature_methods, MODEL_DICT, MODEL_PARAMS, X_train_features_dict, X_test_features_dict, y_train, y_test)


🔎 Running feature extraction and model training loop...


🚀 Training bilstm models...



e:\anaconda3\envs\ml_env_test\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



🚀 Training Bi-LSTM model...
Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.5119 - loss: 0.6933 - val_accuracy: 0.4550 - val_loss: 0.6936
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 0.5025 - loss: 0.6933 - val_accuracy: 0.5450 - val_loss: 0.6832
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.7393 - loss: 0.6345 - val_accuracy: 0.4550 - val_loss: 1.8581
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.8239 - loss: 0.5148 - val_accuracy: 0.6300 - val_loss: 0.7312
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 31s 1s/step - accuracy: 0.9555 - loss: 0.1427 - val_accuracy: 0.6200 - val_loss: 0.9444
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 35s 936ms/step - accuracy: 0.9926 - loss: 0.0266 - val_accuracy: 0.6100 - val_loss: 1.2035
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 1.0000 - loss: 0.0060 - val_accuracy: 0.6100 - val_loss: 1.3358
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 46s 2s/step - accuracy: 1.0000 - loss: 0.0031 - 

KeyboardInterrupt: 

In [ ]:
# predict_general_model(model_name_lst, feature_methods, X_test_features_dict, y_test, trained_model)

In [ ]:
# train_general_model(df_sampled, doc_lst, label_lst, model_name_lst, feature_methods, MODEL_DICT, MODEL_PARAMS, X_train_pca_dict, X_test_pca_dict, y_train, y_test)


🔎 Running feature extraction and model training loop...


🚀 Training lda models...

🔎 Training with Method: tfidf...
🚀 Training new model: LinearDiscriminantAnalysis...
Best hyperparameters: {'shrinkage': 'auto', 'solver': 'lsqr', 'tol': 0.0001}

🎯 Running K-Fold Cross-Validation...


K-Fold Progress: 100%|██████████| 5/5 [00:00<00:00, 16.58it/s]


📊 Average Accuracy: 66%
📊 Average ROC AUC: 65%
📊 Average F1 Score: 68%
📊 Average Precision: 67%
📊 Average Recall: 70%
💾 Model saved to best_lda_tfidf.pkl
📈 Plot saved to best_lda_tfidf.png
📉 Loss plot saved to best_lda_tfidf_loss.png
🔎 Training with Method: count...
🚀 Training new model: LinearDiscriminantAnalysis...
Best hyperparameters: {'shrinkage': 'auto', 'solver': 'lsqr', 'tol': 0.0001}

🎯 Running K-Fold Cross-Validation...


K-Fold Progress: 100%|██████████| 5/5 [00:00<00:00, 16.76it/s]


📊 Average Accuracy: 64%
📊 Average ROC AUC: 63%
📊 Average F1 Score: 69%
📊 Average Precision: 64%
📊 Average Recall: 76%
💾 Model saved to best_lda_count.pkl
📈 Plot saved to best_lda_count.png
📉 Loss plot saved to best_lda_count_loss.png
🔎 Training with Method: word2vec...
🚀 Training new model: LinearDiscriminantAnalysis...
Best hyperparameters: {'shrinkage': None, 'solver': 'lsqr', 'tol': 0.0001}

🎯 Running K-Fold Cross-Validation...


K-Fold Progress: 100%|██████████| 5/5 [00:00<00:00,  5.28it/s]


📊 Average Accuracy: 68%
📊 Average ROC AUC: 67%
📊 Average F1 Score: 70%
📊 Average Precision: 69%
📊 Average Recall: 72%
💾 Model saved to best_lda_word2vec.pkl
📈 Plot saved to best_lda_word2vec.png
📉 Loss plot saved to best_lda_word2vec_loss.png
🔎 Training with Method: glove...
🚀 Training new model: LinearDiscriminantAnalysis...
Best hyperparameters: {'shrinkage': None, 'solver': 'lsqr', 'tol': 0.0001}

🎯 Running K-Fold Cross-Validation...


K-Fold Progress: 100%|██████████| 5/5 [00:00<00:00, 13.55it/s]


📊 Average Accuracy: 65%
📊 Average ROC AUC: 65%
📊 Average F1 Score: 67%
📊 Average Precision: 67%
📊 Average Recall: 68%
💾 Model saved to best_lda_glove.pkl
📈 Plot saved to best_lda_glove.png
📉 Loss plot saved to best_lda_glove_loss.png


In [ ]:
# predict_general_model(model_name_lst, feature_methods, X_test_pca_dict, y_test, trained_model)

🔎 Predicting with Model: lda, Method: tfidf...
Model: lda
Method: tfidf
--------------------------------------------------
Accuracy: 0.5200
Precision: 0.5455
Recall: 0.6168
F1 Score: 0.5789
ROC AUC: 0.5405486885740126
🔎 Predicting with Model: lda, Method: count...
Model: lda
Method: count
--------------------------------------------------
Accuracy: 0.4850
Precision: 0.5156
Recall: 0.6168
F1 Score: 0.5617
ROC AUC: 0.46276756104914074
🔎 Predicting with Model: lda, Method: word2vec...
Model: lda
Method: word2vec
--------------------------------------------------
Accuracy: 0.5700
Precision: 0.6000
Recall: 0.5888
F1 Score: 0.5943
ROC AUC: 0.6128027333936288
🔎 Predicting with Model: lda, Method: glove...
Model: lda
Method: glove
--------------------------------------------------
Accuracy: 0.4900
Precision: 0.5217
Recall: 0.5607
F1 Score: 0.5405
ROC AUC: 0.5018591096372224
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
